# Custom Training with Tensorflow Datasets

- Custm training with tf.data.Dataset in tensorflow is useful when you need more flexibility than what model.fit() provides. It allows fine-grained control over training loops, optimization, and dataset manipulation. 

## Key steps for custom training with tf.data.Dataset

1. Prepare the Dataset
- TensorFlow's tf.data.Dataset API efficiently loads and preprocesses data. You can create a dataset from a numpy array, a CSV file, TFRecord files or other sources. 

In [ ]:
import tensorflow as tf

# Create a simple dataset (features, labels)
features = tf.random.normal([100, 10])  # 100 samples, 10 features each
labels = tf.random.uniform([100, 1], maxval=2, dtype=tf.int32)  # Binary labels

# Convert to tf.data.Dataset
dataset = tf.data.Dataset.from_tensor_slices((features, labels))

# Shuffle, batch, and prefetch for efficiency
batch_size = 32
dataset = dataset.shuffle(buffer_size=100).batch(batch_size).prefetch(tf.data.AUTOTUNE)


2. Define the model
- You can define a custom model using TensorFlow's tf.keras.Model API

In [ ]:
class MyModel(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.dense1 = tf.keras.layers.Dense(64, activation='relu')
        self.dense2 = tf.keras.layers.Dense(1, activation='sigmoid')  # Binary classification

    def call(self, inputs):
        x = self.dense1(inputs)
        return self.dense2(x)

model = MyModel()


3. Define the loss, Optimizer, and metrics

In [ ]:
loss_fn = tf.keras.losses.BinaryCrossentropy()
optimizer = tf.keras.optimizers.Adam()
train_acc_metric = tf.keras.metrics.BinaryAccuracy()


4. Implement the Training Loop
- Instead of model.fit(), you manually control training using a loop with tf.GradientTape

In [ ]:
epochs = 5  # Number of training epochs

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")

    for step, (x_batch, y_batch) in enumerate(dataset):
        with tf.GradientTape() as tape:
            logits = model(x_batch, training=True)  # Forward pass
            loss = loss_fn(y_batch, logits)  # Compute loss

        # Compute gradients and update weights
        grads = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))

        # Update metric
        train_acc_metric.update_state(y_batch, logits)

        if step % 10 == 0:
            print(f"Step {step}, Loss: {loss.numpy():.4f}")

    # Print accuracy at end of each epoch
    train_acc = train_acc_metric.result().numpy()
    print(f"Training Accuracy: {train_acc:.4f}")
    train_acc_metric.reset_states()


# Why use custom training ?
1. Greater Control: Customize gradient updates, loss calculations, and metrics.
2. Custom losses: Implement complex loss functions beyond standard ones. 
3. Flexible Training Steps: Adapt learning rates, early stopping, or other custom logic. 
4. Handles Non-Standard Models: Useful for architectures like GANs, reinforcement learning, or transformer-based models. 